# TP5 : read and write

https://spark.apache.org/docs/latest/sql-data-sources.html

A wide range of data sources can be read
- text
- csv
- parquet
- json

https://mvnrepository.com/artifact/org.apache.spark/spark-avro

- download spark-avro_2.???.jar (following your version of spark and scala) and move it in your */spark/jars

You need to add a config to include spark-avro.

You need to know your spark version, your scale version (use spark UI)

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType 
from pyspark.sql.types import ArrayType, DoubleType, BooleanType
from pyspark.sql.functions import col,array_contains

import findspark
findspark.init()

# spark = SparkSession.builder.appName('TP5').getOrCreate()

# for avro, we need an external modul
spark = SparkSession.builder.appName('TP5').\
config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:3.4.1").\
getOrCreate()

sc = spark.sparkContext

sc

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/11/10 13:40:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=TP5>

## 1. text


### 1.2 Read

https://spark.apache.org/docs/latest/sql-data-sources-text.html

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.read.html#


- Read a file and create a dataframe
    - default separator (what is the by default separator)
    - choose "-" then "." as separator (lineSep=".")


- Read the contents of a folder
    - one row for each line
    - one row for each file (wholetext=True)
    
- It's better with a filename

In [ ]:
import os
# lecture des fichiers du répertoire
# defini un Dataframe
import pyspark.sql.functions
from pyspark.sql import functions as F

# read file "tech/284.txt"
df=spark.read.??
df.show(2)
# choose as separator "." then "-"
df1=spark.read.??
df1.show(2)

# choose as separator "." then "-"
df2=spark.read.??
df2.show(2)

# read the content of tech folder (one row by line 
df3=spark.read.??
df3.show(2)

# then one row by file)
df4=spark.read.??
df4.show(2)

# with the name of the file is usefull
df5 = df.withColumn('filename',F.input_file_name())
df5.show(2, truncate = False)

### 1.2 Write

- write textfiles (you can compare the results between df, df1, df2, df3, df4, df5)
    - without compression
    - with compression (compression="gzip")
    - attention au *mode* : mode("overwrite"),mode("append"),... 
    - *dataframe.write(...).mode(...).text(...) *
    - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.write.html#pyspark.sql.DataFrame.write
    - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.html#pyspark.sql.DataFrameWriter

In [ ]:
# save the dataframe in txt
# without compression
df.write.??
# with compression
df.write.??

# idem with other df
???

## 2. CSV


### 2.1 Read

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.html

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html#pyspark.sql.DataFrameReader.csv

*zipcodes.csv* was modified for this lab.

- Read "./resources/zipcodes.csv"
- Apply *printSchema()* then *show()* to *df*

In [ ]:
df = spark.read.??

df.??
df.??

- Comment and analyze

It's time to define your delimiter

- Take into account the delimiter with *.option(...)* then with *.options(...)*

(*printSchema()* and *show()*)

In [ ]:
# with .option
df2 = spark.read.??

df2.??
df2.??  

#with .options
df3 = spark.read.??

df3.??
df3.??  

- comments and analyze

- oups we have also a header

- modify and complete *.options*

In [ ]:
# with .options
df4 = spark.read.??

df4.??
df4.??

- we can try the option *inferSchema* 

- modify and complete *.options*

In [ ]:
df5 = spark.read.??

df5.??
df5.??

Its better to define our own schema (StrucType)

here an example

```
ourexampleschema = StructType() \
      .add("name_column1",IntegerType(),True) \
      .add("name_column2",StringType(),True) \
      .add("name_column3",DateType(),True) \
      .add("name_column4",TimestampType(),True) \
      .add("name_column5",BooleanType(),True)
```      
      
- adapt this example to your csv file and use *.schema(ourschema)*

In [ ]:
schema = StructType() \
      .add(??
           ????
      ???,True)
      
df6 = spark.???
           
df6.???
df6.??

- DateType and TimestampType have a problem

we need to define dateFormat and TimmeStampFormat: 
    .option("dateFormat","dd/MM/yyyy")
    .option("TimeStampFormat","yyyy-MM-dd HH:mm")
  
(choice the right separator and the right order)

The format depend of your file.

- *DateType* default format is *yyyy-MM-dd*
- *TimestampType* default format is *yyyy-MM-dd HH:mm:ss.SSSS*
  

In [ ]:
df7 = spark.???
           
df7.???
df7.??

- Read a big CSV file and ?

- define the schema

- create a column year and a column month

- I want only Date - Heure, Date, year, month, Consommation brute électricité (MW) - RTE

In [ ]:
from pyspark.sql.functions import year, month, dayofmonth

schema = ???

df_big = 

### 1.2 Write

- write the dataframe as csv

- I'll like to have the previous date format string

(change dateformat, separator)
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.csv.html#pyspark.sql.DataFrameWriter.csv


In [ ]:
# save with by default parameter in forlder ./outputcsv
df7.write.??

# save with by the right option (for the date timestamp) in forlder ./outputcsv1
df7.write.??

- Save df_big (wihout and with .partitionBy  by year and then add by month)

- Compare the content of your folder


In [ ]:
df_big.???

## 3. JSON


### 3.1 Read

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html#pyspark.sql.DataFrameReader.csv

- read *zipcodes.csv* (without options) (12 columns)

- the columns have no name

- what is the type of each columns (use *printSchema()*) ?

- Define the schema

- Define dateFormat and TimeStampFormat

https://spark.apache.org/docs/latest/sql-data-sources-csv.html



In [ ]:
# read json file
df8 = spark.read.???

df8.??
df8.??

The date and timestamp are string

We need to define a schema.

we will use .format("leformat") and .load("pathfichier")

In [ ]:
schema = StructType() \
      .add("id",??? \
      .add(???)

# define .option("dateFormat","???")
# define .option("TimeStampFormat","???")
df9 = spark.read.format(???) \
      .options(???) \
      .schema(???) \
      .load("zipcodes.json")

df9.??
df9.??


- Read the file autre.jason

- *printSchema* and *show*

The schema is more complex.
   
- create and show a dataframe compose of columns : *col_int*, *extra_key* and *key*

In [ ]:
# define the delimiter
df10 = spark.read.json("autre.json")

df10.??
df10.??

df10.select(???).show()

- I only want read the following schema

```
root
 |-- data: struct (nullable = true)
 |    |-- emp_id: string (nullable = true)
 |    |-- awards: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- award_type: string (nullable = true)
 |    |    |    |-- award_name: string (nullable = true)
 ```

- define the right schema then read the test_emp.jason

- use *.printSchema()*

- show the columns emp_name and award_name


In [ ]:
schema = ???
 
df11 = spark.read.json(path= "test_emp.json", schema=schema, multiLine=True)
df11.show(truncate=False)

df11.???
df11.???

### 3.2 Write

- write textfiles on  outputjason
    - attention au *mode*
    - *dataframe.write(...).mode(...).text(...) *
    - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.write.html#pyspark.sql.DataFrame.write
    - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.html#pyspark.sql.DataFrameWriter

In [10]:
df.write.???

## 4. Parquet and Avro

Avro and Parquet and Big Data ?

useful or useless ?

https://spark.apache.org/docs/latest/sql-data-sources-parquet.html

### 4.1 Parquet

In [11]:
peopleDF = spark.read.json("people.json")

# DataFrames can be saved as Parquet files, maintaining the schema information.
peopleDF.write.mode("overwrite").parquet("people.parquet")

# Read in the Parquet file created above.
# Parquet files are self-describing so the schema is preserved.
# The result of loading a parquet file is also a DataFrame.
parquetFile = spark.read.parquet("people.parquet")
parquetFile.show()
# Parquet files can also be used to create a temporary view and then used in SQL statements.
parquetFile.createOrReplaceTempView("parquetFile")
teenagers = spark.sql("SELECT * FROM parquetFile WHERE age >= 13 AND age <= 19")
teenagers.show()
# +------+
# |  name|
# +------+
# |Justin|
# +------+

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

+---+------+
|age|  name|
+---+------+
| 19|Justin|
+---+------+



In [ ]:
data = [('James','Smith','M',3000),
  ('Anna','Rose','F',4100),
  ('Robert','Williams','M',6200), 
]

columns = ["prenom","nom","genre","salaire"]


df = spark.createDataFrame(data=data, schema = columns)
df.show()

# partitionBy() Example

df.write.option("header",True) \
        .partitionBy("genre") \
        .mode("overwrite") \
        .parquet("/tmp/gendertest")


df.write.option("header",True) \
        .partitionBy("genre","salaire") \
        .mode("overwrite") \
        .parquer("/tmp/gendersalarytest")

### 4.2 Parquet

- use *.format("avro")* and *.save(...)* or *.load(...)*

In [ ]:
peopleDF = spark.read.json("people.json")

peopleDF.write.format(???).mode("overwrite").save(???)
# Read in the Parquet file created above.
# Parquet files are self-describing so the schema is preserved.
# The result of loading a parquet file is also a DataFrame.
parquetFile = spark.read.format(???).load(???)
parquetFile.show()
# Parquet files can also be used to create a temporary view and then used in SQL statements.
parquetFile.createOrReplaceTempView("parquetFile")
teenagers = spark.sql("SELECT * FROM parquetFile WHERE age >= 13 AND age <= 19")
teenagers.show()

In [ ]:
data = [('James','Smith','M',3000),
  ('Anna','Rose','F',4100),
  ('Robert','Williams','M',6200), 
]

columns = ["prenom","nom","genre","salaire"]
df = spark.createDataFrame(data=data, schema = columns)
df.show()


# partitionBy()

# Partitionner en fonction du genre
df.write.option("header",True) \
        .partitionBy(??) \
        .mode("overwrite") \
        .parquet("??")


# Partitionner en fonction du genre puis du salaire
df.write.option("header",True) \
        .partitionBy(???) \
        .mode("overwrite") \
        .parquet(??)
                 

# A partir du répertoire précédent, lire les données de genre M et du salaire 4100


# A partir du répertoire précédent, lire les données de genre M